# Post Pandemic Regime Shifts in Labor Market: Data Cleaning and Merge

## Purpose

## Research Context

## Imports and Configuration

In [47]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

In [48]:
repo_root = Path.home() / "Documents" / "Coding" / "ML_LaborTightnessRegimeShift"
# repo_root = Path(r"C:\Users\sumai\Documents\ML_LaborTightnessRegimeShift")

data_root = repo_root / "data"
raw_dir = data_root / "raw"
processed_dir = data_root / "processed"

In [49]:
fred_dir = raw_dir / "fred"
external_dir = raw_dir / "external_sources"
key_raw_dir = external_dir / "key_raw_tables"

fred_raw_in = fred_dir / "fred_raw_monthly.csv"
fred_meta_in = fred_dir / "fred_metadata.csv"
fred_failed_in = fred_dir / "fred_failed_series.csv"
fred_missing_in = fred_dir / "fred_missing_summary.csv"

atl_raw_in = key_raw_dir / "atlanta_fed_wgt_overall_raw.csv"
atl_12ma_raw_in = key_raw_dir / "atlanta_fed_wgt_overall_12ma_raw.csv"
phl_ruc_raw_in = key_raw_dir / "philly_fed_ruc_vintages_raw.csv"
phl_cpi_raw_in = key_raw_dir / "philly_fed_cpi_vintages_raw.csv"
spf_raw_in = key_raw_dir / "philly_fed_spf_inflation_raw.csv"

fred_clean_out = processed_dir / "01_fred_clean_monthly.csv"
atl_clean_out = processed_dir / "02_atl_wage_tracker_clean.csv"
phl_ruc_clean_out = processed_dir / "03_phl_ruc_latest_clean.csv"
phl_cpi_clean_out = processed_dir / "04_phl_cpi_latest_clean.csv"
spf_clean_out = processed_dir / "05_spf_inflation_clean.csv"
base_out = processed_dir / "06_macro_labor_base.csv"
model_ready_out = processed_dir / "07_model_ready_monthly.csv"

audit_out = processed_dir / "08_cleaning_audit_summary.csv"
missing_out = processed_dir / "08_cleaning_missing_summary.csv"
schema_out = processed_dir / "08_cleaning_schema_summary.csv"
date_out = processed_dir / "08_cleaning_date_summary.csv"

processed_dir.mkdir(parents=True, exist_ok=True)

In [50]:
pandemic_start = pd.Timestamp("2020-03-01")
post_pandemic_start = pd.Timestamp("2021-04-01")

## Load Raw Files

In [51]:
df_fred = pd.read_csv(fred_raw_in, low_memory=False)
df_fred_meta = pd.read_csv(fred_meta_in, low_memory=False)
df_fred_failed = pd.read_csv(fred_failed_in, low_memory=False)
df_fred_missing = pd.read_csv(fred_missing_in, low_memory=False)

df_atl_raw = pd.read_csv(atl_raw_in, low_memory=False)
df_atl_12ma_raw = pd.read_csv(atl_12ma_raw_in, low_memory=False)
df_phl_ruc_raw = pd.read_csv(phl_ruc_raw_in, low_memory=False)
df_phl_cpi_raw = pd.read_csv(phl_cpi_raw_in, low_memory=False)
df_spf_raw = pd.read_csv(spf_raw_in, low_memory=False)

print("Loaded Shapes")
print(". . .")
print("FRED Raw:", df_fred.shape)
print("FRED Metadata:", df_fred_meta.shape)
print("FRED Failed:", df_fred_failed.shape)
print("FRED Missing:", df_fred_missing.shape)
print("ATL Raw:", df_atl_raw.shape)
print("ATL 12MA Raw:", df_atl_12ma_raw.shape)
print("PHL RUC Raw:", df_phl_ruc_raw.shape)
print("PHL CPI Raw:", df_phl_cpi_raw.shape)
print("SPF Raw:", df_spf_raw.shape)

Loaded Shapes
. . .
FRED Raw: (315, 67)
FRED Metadata: (66, 14)
FRED Failed: (1, 3)
FRED Missing: (66, 7)
ATL Raw: (351, 17)
ATL 12MA Raw: (352, 4)
PHL RUC Raw: (949, 243)
PHL CPI Raw: (949, 243)
SPF Raw: (225, 5)


## Functions for Cleaning

In [52]:
def clean_name(text: str) -> str:
    text = str(text).strip().lower()
    text = text.replace("%", "pct")
    text = text.replace("&", "and")
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", "_", text)
    text = re.sub(r"_+", "_", text)
    return text.strip("_")


def make_unique(names):
    seen = {}
    out = []

    for name in names:
        if name not in seen:
            seen[name] = 0
            out.append(name)
        else:
            seen[name] += 1
            out.append(f"{name}_{seen[name]}")

    return out


def parse_month_text(text):
    return pd.to_datetime(str(text), format="%Y:%m", errors="coerce")


def to_numeric_frame(df: pd.DataFrame, skip_cols=None) -> pd.DataFrame:
    if skip_cols is None:
        skip_cols = []

    df = df.copy()

    for col in df.columns:
        if col in skip_cols:
            continue

        df[col] = (
            df[col]
            .replace(".", np.nan)
            .replace("...", np.nan)
            .replace("NA", np.nan)
            .replace("N.A.", np.nan)
        )

        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


def latest_value(row: pd.Series):
    row = row.dropna()
    if len(row) == 0:
        return np.nan
    return row.iloc[-1]


def latest_vintage(row: pd.Series):
    row = row.dropna()
    if len(row) == 0:
                return np.nan
    return row.index[-1]

In [53]:
df_fred.columns = (
    df_fred.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

df_fred["date"] = pd.to_datetime(df_fred["date"], errors="coerce")

fred_num_cols = [c for c in df_fred.columns if c != "date"]
for col in fred_num_cols:
    df_fred[col] = pd.to_numeric(df_fred[col], errors="coerce")

df_fred = (
    df_fred
    .dropna(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

df_fred.to_csv(fred_clean_out, index=False)

print("\nFRED after the cleaning")
print(". . .")
print("Shape:", df_fred.shape)
print("Date Range:", df_fred["date"].min(), "to", df_fred["date"].max())
print("Saved:", fred_clean_out)


FRED after the cleaning
. . .
Shape: (315, 67)
Date Range: 2000-01-01 00:00:00 to 2026-03-01 00:00:00
Saved: /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/processed/01_fred_clean_monthly.csv


## Atlanta Fed Wage Tracker

In [54]:
df_atl_header = df_atl_raw.iloc[0].copy()

atl_cols = ["date"] + [clean_name(v) for v in df_atl_header.iloc[1:].tolist()]
atl_cols = make_unique(atl_cols)

df_atl = df_atl_raw.iloc[1:].copy()
df_atl.columns = atl_cols

df_atl["date"] = pd.to_datetime(df_atl["date"], errors="coerce")
df_atl = df_atl.dropna(subset=["date"]).copy()
df_atl = to_numeric_frame(df_atl, skip_cols=["date"])

atl_rename = {
    "overall": "atl_wage_growth_overall_3mma",
    "services": "atl_wage_growth_services_3mma",
    "fulltime": "atl_wage_growth_fulltime_3mma",
    "college_degree": "atl_wage_growth_college_degree_3mma",
    "age_2554": "atl_wage_growth_age_2554_3mma",
    "female": "atl_wage_growth_female_3mma",
    "male": "atl_wage_growth_male_3mma",
    "job_stayer": "atl_wage_growth_job_stayer_3mma",
    "job_switcher": "atl_wage_growth_job_switcher_3mma",
    "paid_hourly": "atl_wage_growth_paid_hourly_3mma",
    "overall_weighted": "atl_wage_growth_overall_weighted_3mma",
    "overall_weighted_97": "atl_wage_growth_overall_weighted_97_3mma",
    "overall_weekly_basis": "atl_wage_growth_overall_weekly_basis_3mma",
    "overall_2520_trimmed_mean": "atl_wage_growth_trimmed_mean_3mma",
    "lower_12_of_wage_distn": "atl_wage_growth_lower_half_3mma",
    "upper_12_of_wage_distn": "atl_wage_growth_upper_half_3mma",
}

df_atl = df_atl.rename(columns={k: v for k, v in atl_rename.items() if k in df_atl.columns})
df_atl = df_atl.sort_values("date").reset_index(drop=True)

df_atl_12ma_header = df_atl_12ma_raw.iloc[1].copy()

atl_12ma_cols = ["date"] + [clean_name(v) for v in df_atl_12ma_header.iloc[1:].tolist()]
atl_12ma_cols = make_unique(atl_12ma_cols)

df_atl_12ma = df_atl_12ma_raw.iloc[2:].copy()
df_atl_12ma.columns = atl_12ma_cols

df_atl_12ma["date"] = pd.to_datetime(df_atl_12ma["date"], errors="coerce")
df_atl_12ma = df_atl_12ma.dropna(subset=["date"]).copy()
df_atl_12ma = to_numeric_frame(df_atl_12ma, skip_cols=["date"])

atl_12ma_rename = {
    "overall": "atl_wage_growth_overall_12mma",
    "overall_weighted": "atl_wage_growth_overall_weighted_12mma",
    "overall_weighted_97": "atl_wage_growth_overall_weighted_97_12mma",
}

df_atl_12ma = df_atl_12ma.rename(columns={k: v for k, v in atl_12ma_rename.items() if k in df_atl_12ma.columns})
df_atl_12ma = df_atl_12ma.sort_values("date").reset_index(drop=True)

df_atl_clean = (
    df_atl
    .merge(df_atl_12ma, how="outer", on="date")
    .sort_values("date")
    .reset_index(drop=True)
)

df_atl_clean.to_csv(atl_clean_out, index=False)

print("\nATL after Clean")
print(". . .")
print("Shape:", df_atl_clean.shape)
print("Date Range:", df_atl_clean["date"].min(), "to", df_atl_clean["date"].max())
print("Saved:", atl_clean_out)


ATL after Clean
. . .
Shape: (350, 20)
Date Range: 1997-01-01 00:00:00 to 2026-02-01 00:00:00
Saved: /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/processed/02_atl_wage_tracker_clean.csv


## Philadelphia Fed RUC Vintages Cleaning

In [55]:
df_phl_ruc = df_phl_ruc_raw.copy()
df_phl_ruc.columns = [clean_name(c) for c in df_phl_ruc.columns]
df_phl_ruc = df_phl_ruc.rename(columns={"date": "date_text"})
df_phl_ruc["date"] = df_phl_ruc["date_text"].apply(parse_month_text)

phl_ruc_val_cols = [c for c in df_phl_ruc.columns if c not in ["date_text", "date"]]
df_phl_ruc[phl_ruc_val_cols] = df_phl_ruc[phl_ruc_val_cols].apply(pd.to_numeric, errors="coerce")

df_phl_ruc_clean = df_phl_ruc[["date"]].copy()
df_phl_ruc_clean["phl_ruc_latest"] = df_phl_ruc[phl_ruc_val_cols].apply(latest_value, axis=1)
df_phl_ruc_clean["phl_ruc_latest_vintage"] = df_phl_ruc[phl_ruc_val_cols].apply(latest_vintage, axis=1)

df_phl_ruc_clean = (
    df_phl_ruc_clean
    .dropna(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

df_phl_ruc_clean.to_csv(phl_ruc_clean_out, index=False)

print("\nPHL RUC Clean")
print(". . .")
print("Shape:", df_phl_ruc_clean.shape)
print("Date Range:", df_phl_ruc_clean["date"].min(), "to", df_phl_ruc_clean["date"].max())
print("Saved:", phl_ruc_clean_out)


PHL RUC Clean
. . .
Shape: (949, 3)
Date Range: 1947-01-01 00:00:00 to 2026-01-01 00:00:00
Saved: /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/processed/03_phl_ruc_latest_clean.csv


In [56]:
df_phl_cpi = df_phl_cpi_raw.copy()
df_phl_cpi.columns = [clean_name(c) for c in df_phl_cpi.columns]
df_phl_cpi = df_phl_cpi.rename(columns={"date": "date_text"})
df_phl_cpi["date"] = df_phl_cpi["date_text"].apply(parse_month_text)

phl_cpi_val_cols = [c for c in df_phl_cpi.columns if c not in ["date_text", "date"]]
df_phl_cpi[phl_cpi_val_cols] = df_phl_cpi[phl_cpi_val_cols].apply(pd.to_numeric, errors="coerce")

df_phl_cpi_clean = df_phl_cpi[["date"]].copy()
df_phl_cpi_clean["phl_cpi_latest"] = df_phl_cpi[phl_cpi_val_cols].apply(latest_value, axis=1)
df_phl_cpi_clean["phl_cpi_latest_vintage"] = df_phl_cpi[phl_cpi_val_cols].apply(latest_vintage, axis=1)

df_phl_cpi_clean = (
    df_phl_cpi_clean
    .dropna(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

df_phl_cpi_clean.to_csv(phl_cpi_clean_out, index=False)

print("\nPHL CPI ")
print(". . .")
print("Shape:", df_phl_cpi_clean.shape)
print("Date Range:", df_phl_cpi_clean["date"].min(), "to", df_phl_cpi_clean["date"].max())
print("Saved:", phl_cpi_clean_out)


PHL CPI 
. . .
Shape: (949, 3)
Date Range: 1947-01-01 00:00:00 to 2026-01-01 00:00:00
Saved: /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/processed/04_phl_cpi_latest_clean.csv


## SPF Inflation

In [57]:
df_spf = df_spf_raw.copy()

df_spf.columns = (
    df_spf.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

for col in df_spf.columns:
    df_spf[col] = pd.to_numeric(df_spf[col], errors="coerce")

df_spf["year"] = df_spf["year"].astype("Int64")
df_spf["quarter"] = df_spf["quarter"].astype("Int64")

df_spf["quarter_text"] = (
    df_spf["year"].astype("string")
    + "Q"
    + df_spf["quarter"].astype("string")
)

df_spf["date"] = (
    pd.PeriodIndex(df_spf["quarter_text"], freq="Q")
    .to_timestamp(how="start")
)

spf_rename = {
    "infpgdp1yr": "spf_gdp_inflation_1y",
    "infcpi1yr": "spf_cpi_inflation_1y",
    "infcpi10yr": "spf_cpi_inflation_10y",
}

df_spf = df_spf.rename(columns={k: v for k, v in spf_rename.items() if k in df_spf.columns})

spf_keep_cols = ["date", "year", "quarter"] + [c for c in spf_rename.values() if c in df_spf.columns]

df_spf_clean = (
    df_spf[spf_keep_cols]
    .dropna(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

df_spf_clean.to_csv(spf_clean_out, index=False)
print("\nSPF Clean")
print(". . .")
print("Shape:", df_spf_clean.shape)
print("Date Range:", df_spf_clean["date"].min(), "to", df_spf_clean["date"].max())
print("Saved:", spf_clean_out)


SPF Clean
. . .
Shape: (225, 6)
Date Range: 1970-01-01 00:00:00 to 2026-01-01 00:00:00
Saved: /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/processed/05_spf_inflation_clean.csv


In [58]:
date_min = min(
    df_fred["date"].min(),
    df_atl_clean["date"].min(),
    df_phl_ruc_clean["date"].min(),
    df_phl_cpi_clean["date"].min(),
    df_spf_clean["date"].min(),
)

date_max = max(
    df_fred["date"].max(),
    df_atl_clean["date"].max(),
    df_phl_ruc_clean["date"].max(),
    df_phl_cpi_clean["date"].max(),
    df_spf_clean["date"].max(),
)

df_base = pd.DataFrame({
    "date": pd.date_range(date_min, date_max, freq="MS")
})

df_base = (
    df_base
    .merge(df_fred, how="left", on="date")
    .merge(df_atl_clean, how="left", on="date")
    .merge(df_phl_ruc_clean, how="left", on="date")
    .merge(df_phl_cpi_clean, how="left", on="date")
    .merge(
        df_spf_clean[["date"] + [c for c in df_spf_clean.columns if c.startswith("spf_")]],
        how="left",
        on="date",
    )
    .sort_values("date")
    .reset_index(drop=True)
)

spf_cols = [c for c in df_base.columns if c.startswith("spf_")]
if spf_cols:
    df_base[spf_cols] = df_base[spf_cols].ffill()

In [59]:
df_base["year"] = df_base["date"].dt.year
df_base["quarter"] = df_base["date"].dt.quarter
df_base["month"] = df_base["date"].dt.month

df_base["is_pre_pandemic"] = (df_base["date"] < pandemic_start).astype(int)
df_base["is_pandemic_transition"] = (
    (df_base["date"] >= pandemic_start) &
    (df_base["date"] < post_pandemic_start)
).astype(int)
df_base["is_post_pandemic"] = (df_base["date"] >= post_pandemic_start).astype(int)

df_base["regime_period"] = np.select(
    [
        df_base["date"] < pandemic_start,
        (df_base["date"] >= pandemic_start) & (df_base["date"] < post_pandemic_start),
        df_base["date"] >= post_pandemic_start,
    ],
    [
        "pre_pandemic",
        "pandemic_transition",
        "post_pandemic",
    ],
    default="unknown",
)

In [60]:
feature_df = pd.DataFrame(index=df_base.index)

if {"job_openings_level", "unemployment_level"}.issubset(df_base.columns):
    feature_df["vacancy_unemployment_ratio"] = np.where(
        df_base["unemployment_level"] > 0,
        df_base["job_openings_level"] / df_base["unemployment_level"],
        np.nan,
    )

if {"job_openings_level", "payrolls_total_nonfarm"}.issubset(df_base.columns):
    feature_df["job_openings_share_payrolls"] = np.where(
        df_base["payrolls_total_nonfarm"] > 0,
        100 * df_base["job_openings_level"] / df_base["payrolls_total_nonfarm"],
        np.nan,
    )

if {"quits_rate", "layoffs_discharges_rate"}.issubset(df_base.columns):
    feature_df["quits_layoffs_ratio"] = np.where(
        df_base["layoffs_discharges_rate"] > 0,
        df_base["quits_rate"] / df_base["layoffs_discharges_rate"],
        np.nan,
    )

if "cpi_all_items" in df_base.columns:
    feature_df["cpi_all_items_yoy"] = 100 * df_base["cpi_all_items"].pct_change(12)

if "cpi_core" in df_base.columns:
    feature_df["cpi_core_yoy"] = 100 * df_base["cpi_core"].pct_change(12)

if "pce_price_index" in df_base.columns:
    feature_df["pce_price_index_yoy"] = 100 * df_base["pce_price_index"].pct_change(12)

if "pce_core_price_index" in df_base.columns:
    feature_df["pce_core_price_index_yoy"] = 100 * df_base["pce_core_price_index"].pct_change(12)

if "avg_hourly_earnings_total_private" in df_base.columns:
    feature_df["avg_hourly_earnings_total_private_yoy"] = (
        100 * df_base["avg_hourly_earnings_total_private"].pct_change(12)
    )

if {"michigan_inflation_1y", "cpi_all_items_yoy"}.issubset(feature_df.columns.union(df_base.columns)):
    feature_df["inflation_expectations_gap_1y"] = (
        df_base["michigan_inflation_1y"] - feature_df["cpi_all_items_yoy"]
    )

if {"u6_unemployment_rate", "unemployment_rate"}.issubset(df_base.columns):
    feature_df["u6_u3_gap"] = df_base["u6_unemployment_rate"] - df_base["unemployment_rate"]

if {"treasury_10y", "treasury_3m"}.issubset(df_base.columns):
    feature_df["term_spread_10y_3m"] = df_base["treasury_10y"] - df_base["treasury_3m"]

if {"baa_corporate_yield", "aaa_corporate_yield"}.issubset(df_base.columns):
    feature_df["baa_aaa_spread"] = df_base["baa_corporate_yield"] - df_base["aaa_corporate_yield"]

if {"atl_wage_growth_overall_3mma", "avg_hourly_earnings_total_private_yoy"}.issubset(
    set(df_base.columns).union(set(feature_df.columns))
):
    feature_df["atl_ahe_gap"] = (
        df_base["atl_wage_growth_overall_3mma"] - feature_df["avg_hourly_earnings_total_private_yoy"]
    )


In [61]:
df_base = pd.concat([df_base, feature_df], axis=1)
df_base = df_base.copy()

df_base.to_csv(base_out, index=False)

print("\nBase Table")
print(". . .")
print("Shape:", df_base.shape)
print("Date Range:", df_base["date"].min(), "to", df_base["date"].max())
print("Saved:", base_out)


Base Table
. . .
Shape: (951, 113)
Date Range: 1947-01-01 00:00:00 to 2026-03-01 00:00:00
Saved: /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/processed/06_macro_labor_base.csv


In [62]:
drop_model_cols = [
    "phl_ruc_latest_vintage",
    "phl_cpi_latest_vintage",
]

drop_model_cols = [c for c in drop_model_cols if c in df_base.columns]

df_model = df_base.drop(columns=drop_model_cols).copy()
df_model.to_csv(model_ready_out, index=False)

print("\nModel-Ready Table")
print(". . .")
print("Shape:", df_model.shape)
print("Date Range:", df_model["date"].min(), "to", df_model["date"].max())
print("Saved:", model_ready_out)


Model-Ready Table
. . .
Shape: (951, 111)
Date Range: 1947-01-01 00:00:00 to 2026-03-01 00:00:00
Saved: /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/processed/07_model_ready_monthly.csv


In [63]:
audit_rows = [
    {
        "table_name": "fred_clean",
        "rows": len(df_fred),
        "cols": df_fred.shape[1],
        "date_min": df_fred["date"].min(),
        "date_max": df_fred["date"].max(),
        "missing_date": df_fred["date"].isna().sum(),
        "duplicate_date": df_fred["date"].duplicated().sum(),
    },
    {
        "table_name": "atl_clean",
        "rows": len(df_atl_clean),
        "cols": df_atl_clean.shape[1],
        "date_min": df_atl_clean["date"].min(),
        "date_max": df_atl_clean["date"].max(),
        "missing_date": df_atl_clean["date"].isna().sum(),
        "duplicate_date": df_atl_clean["date"].duplicated().sum(),
    },
    {
        "table_name": "phl_ruc_clean",
        "rows": len(df_phl_ruc_clean),
        "cols": df_phl_ruc_clean.shape[1],
        "date_min": df_phl_ruc_clean["date"].min(),
        "date_max": df_phl_ruc_clean["date"].max(),
        "missing_date": df_phl_ruc_clean["date"].isna().sum(),
        "duplicate_date": df_phl_ruc_clean["date"].duplicated().sum(),
    },
    {
        "table_name": "phl_cpi_clean",
        "rows": len(df_phl_cpi_clean),
        "cols": df_phl_cpi_clean.shape[1],
        "date_min": df_phl_cpi_clean["date"].min(),
        "date_max": df_phl_cpi_clean["date"].max(),
        "missing_date": df_phl_cpi_clean["date"].isna().sum(),
        "duplicate_date": df_phl_cpi_clean["date"].duplicated().sum(),
    },
    {
                "table_name": "spf_clean",
        "rows": len(df_spf_clean),
        "cols": df_spf_clean.shape[1],
        "date_min": df_spf_clean["date"].min(),
        "date_max": df_spf_clean["date"].max(),
        "missing_date": df_spf_clean["date"].isna().sum(),
        "duplicate_date": df_spf_clean["date"].duplicated().sum(),
    },
    {
        "table_name": "macro_labor_base",
        "rows": len(df_base),
        "cols": df_base.shape[1],
        "date_min": df_base["date"].min(),
        "date_max": df_base["date"].max(),
        "missing_date": df_base["date"].isna().sum(),
        "duplicate_date": df_base["date"].duplicated().sum(),
    },
    {
        "table_name": "model_ready",
        "rows": len(df_model),
        "cols": df_model.shape[1],
        "date_min": df_model["date"].min(),
        "date_max": df_model["date"].max(),
        "missing_date": df_model["date"].isna().sum(),
        "duplicate_date": df_model["date"].duplicated().sum(),
    },
]

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(audit_out, index=False)

In [64]:
schema_frames = []
for name, frame in [
    ("fred_clean", df_fred),
    ("atl_clean", df_atl_clean),
    ("phl_ruc_clean", df_phl_ruc_clean),
    ("phl_cpi_clean", df_phl_cpi_clean),
    ("spf_clean", df_spf_clean),
    ("macro_labor_base", df_base),
    ("model_ready", df_model),
]:
    temp = pd.DataFrame({
        "table_name": name,
        "column": frame.columns,
        "dtype": [str(frame[c].dtype) for c in frame.columns],
        "nunique": [frame[c].nunique(dropna=True) for c in frame.columns],
    })
    schema_frames.append(temp)

schema_df = pd.concat(schema_frames, ignore_index=True)
schema_df.to_csv(schema_out, index=False)

In [65]:
missing_frames = []
for name, frame in [
    ("fred_clean", df_fred),
    ("atl_clean", df_atl_clean),
    ("phl_ruc_clean", df_phl_ruc_clean),
    ("phl_cpi_clean", df_phl_cpi_clean),
    ("spf_clean", df_spf_clean),
    ("macro_labor_base", df_base),
    ("model_ready", df_model),
]:
    temp = pd.DataFrame({
        "table_name": name,
        "column": frame.columns,
        "dtype": [str(frame[c].dtype) for c in frame.columns],
        "missing_count": [frame[c].isna().sum() for c in frame.columns],
    })
    temp["missing_pct"] = 100 * temp["missing_count"] / len(frame)
    missing_frames.append(temp)

missing_df = (
    pd.concat(missing_frames, ignore_index=True)
    .sort_values(["table_name", "missing_count", "column"], ascending=[True, False, True])
    .reset_index(drop=True)
)

missing_df.to_csv(missing_out, index=False)

In [66]:
date_df = pd.DataFrame({
    "table_name": [
        "fred_clean",
        "atl_clean",
        "phl_ruc_clean",
        "phl_cpi_clean",
        "spf_clean",
        "macro_labor_base",
        "model_ready",
    ],
    "date_min": [
        df_fred["date"].min(),
        df_atl_clean["date"].min(),
        df_phl_ruc_clean["date"].min(),
        df_phl_cpi_clean["date"].min(),
        df_spf_clean["date"].min(),
        df_base["date"].min(),
        df_model["date"].min(),
    ],
    "date_max": [
        df_fred["date"].max(),
        df_atl_clean["date"].max(),
        df_phl_ruc_clean["date"].max(),
        df_phl_cpi_clean["date"].max(),
        df_spf_clean["date"].max(),
        df_base["date"].max(),
        df_model["date"].max(),
    ],
    "n_dates": [
        df_fred["date"].nunique(),
        df_atl_clean["date"].nunique(),
        df_phl_ruc_clean["date"].nunique(),
        df_phl_cpi_clean["date"].nunique(),
        df_spf_clean["date"].nunique(),
        df_base["date"].nunique(),
        df_model["date"].nunique(),
    ],
})

date_df.to_csv(date_out, index=False)

print("\nAudit Outputs")
print(". . .")
print("Audit:", audit_out)
print("Missing:", missing_out)
print("Schema:", schema_out)
print("Date:", date_out)


Audit Outputs
. . .
Audit: /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/processed/08_cleaning_audit_summary.csv
Missing: /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/processed/08_cleaning_missing_summary.csv
Schema: /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/processed/08_cleaning_schema_summary.csv
Date: /Users/Sumaitat/Documents/Coding/ML_LaborTightnessRegimeShift/data/processed/08_cleaning_date_summary.csv


In [67]:
required_cols = [
    "date",
    "job_openings_level",
    "hires_rate",
    "quits_rate",
    "unemployment_rate",
    "cpi_all_items",
    "cpi_core",
    "pce_price_index",
    "pce_core_price_index",
    "fed_funds_rate",
    "regime_period",
]

missing_required = [c for c in required_cols if c not in df_model.columns]

print("\nFinal Validation")
print(". . .")
print("Missing Required Columns:", missing_required)
print("Failed FRED Series Count:", len(df_fred_failed))
print("Model-Ready Rows:", len(df_model))
print("Model-Ready Columns:", df_model.shape[1])

assert len(missing_required) == 0, f"Missing required columns: {missing_required}"
assert df_model["date"].isna().sum() == 0
assert df_model["date"].duplicated().sum() == 0

display(audit_df)
display(missing_df.head(50))
display(schema_df.head(50))
display(date_df)
display(df_model.head(12))


Final Validation
. . .
Missing Required Columns: []
Failed FRED Series Count: 1
Model-Ready Rows: 951
Model-Ready Columns: 111


,table_name,rows,cols,date_min,date_max,missing_date,duplicate_date
0,fred_clean,315,67,2000-01-01,2026-03-01,0,0
1,atl_clean,350,20,1997-01-01,2026-02-01,0,0
2,phl_ruc_clean,949,3,1947-01-01,2026-01-01,0,0
3,phl_cpi_clean,949,3,1947-01-01,2026-01-01,0,0
4,spf_clean,225,6,1970-01-01,2026-01-01,0,0
5,macro_labor_base,951,113,1947-01-01,2026-03-01,0,0
6,model_ready,951,111,1947-01-01,2026-03-01,0,0


,table_name,column,dtype,missing_count,missing_pct
0,atl_clean,atl_wage_growth_overall_12mma,float64,12,3.428571
1,atl_clean,atl_wage_growth_overall_weighted_12mma,float64,12,3.428571
2,atl_clean,atl_wage_growth_overall_weighted_97_12mma,float64,12,3.428571
3,atl_clean,atl_wage_growth_age_2554_3mma,float64,3,0.857143
4,atl_clean,atl_wage_growth_college_degree_3mma,float64,3,0.857143
5,atl_clean,atl_wage_growth_female_3mma,float64,3,0.857143
6,atl_clean,atl_wage_growth_fulltime_3mma,float64,3,0.857143
7,atl_clean,atl_wage_growth_job_stayer_3mma,float64,3,0.857143
8,atl_clean,atl_wage_growth_job_switcher_3mma,float64,3,0.857143
9,atl_clean,atl_wage_growth_lower_half_3mma,float64,3,0.857143


,table_name,column,dtype,nunique
0,fred_clean,date,datetime64[us],315
1,fred_clean,job_openings_level,float64,298
2,fred_clean,hires_rate,float64,21
3,fred_clean,hires_level,float64,281
4,fred_clean,quits_rate,float64,19
5,fred_clean,quits_level,float64,286
6,fred_clean,total_separations_rate,float64,16
7,fred_clean,total_separations_level,float64,272
8,fred_clean,layoffs_discharges_level,float64,240
9,fred_clean,layoffs_discharges_rate,float64,14


,table_name,date_min,date_max,n_dates
0,fred_clean,2000-01-01,2026-03-01,315
1,atl_clean,1997-01-01,2026-02-01,350
2,phl_ruc_clean,1947-01-01,2026-01-01,949
3,phl_cpi_clean,1947-01-01,2026-01-01,949
4,spf_clean,1970-01-01,2026-01-01,225
5,macro_labor_base,1947-01-01,2026-03-01,951
6,model_ready,1947-01-01,2026-03-01,951


,date,job_openings_level,hires_rate,hires_level,quits_rate,quits_level,total_separations_rate,total_separations_level,layoffs_discharges_level,layoffs_discharges_rate,unemployment_rate,unemployment_level,payrolls_total_nonfarm,payrolls_total_private,payrolls_manufacturing,payrolls_service_providing,payrolls_construction,prime_age_lfpr,labor_force_participation_rate,employment_population_ratio,u6_unemployment_rate,prime_age_unemployment_rate,average_weeks_unemployed,avg_hourly_earnings_total_private,avg_weekly_earnings_total_private,avg_weekly_hours_total_private,avg_weekly_earnings_manufacturing,avg_hourly_earnings_manufacturing,cpi_all_items,cpi_core,cpi_all_items_less_shelter,cpi_services_less_rent_of_shelter,cpi_services_less_energy_services,cpi_shelter,pce_price_index,pce_core_price_index,personal_saving_rate,real_disposable_personal_income,real_personal_income,real_personal_income_less_transfers,retail_sales_advance,real_retail_sales,industrial_production,industrial_production_manufacturing,capacity_utilization,housing_starts,building_permits,new_one_family_houses_sold,consumer_sentiment,durable_goods_orders,...,high_yield_oas,bbb_oas,breakeven_5y,forward_5y5y_inflation,michigan_inflation_1y,recession_indicator,atl_wage_growth_overall_3mma,atl_wage_growth_services_3mma,atl_wage_growth_fulltime_3mma,atl_wage_growth_college_degree_3mma,atl_wage_growth_age_2554_3mma,atl_wage_growth_female_3mma,atl_wage_growth_male_3mma,atl_wage_growth_job_stayer_3mma,atl_wage_growth_job_switcher_3mma,atl_wage_growth_paid_hourly_3mma,atl_wage_growth_overall_weighted_3mma,atl_wage_growth_overall_weighted_97_3mma,atl_wage_growth_overall_weekly_basis_3mma,atl_wage_growth_trimmed_mean_3mma,atl_wage_growth_lower_half_3mma,atl_wage_growth_upper_half_3mma,atl_wage_growth_overall_12mma,atl_wage_growth_overall_weighted_12mma,atl_wage_growth_overall_weighted_97_12mma,phl_ruc_latest,phl_cpi_latest,spf_gdp_inflation_1y,spf_cpi_inflation_1y,spf_cpi_inflation_10y,year,quarter,month,is_pre_pandemic,is_pandemic_transition,is_post_pandemic,regime_period,vacancy_unemployment_ratio,job_openings_share_payrolls,quits_layoffs_ratio,cpi_all_items_yoy,cpi_core_yoy,pce_price_index_yoy,pce_core_price_index_yoy,avg_hourly_earnings_total_private_yoy,inflation_expectations_gap_1y,u6_u3_gap,term_spread_10y_3m,baa_aaa_spread,atl_ahe_gap
0,1947-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.48,NaN,NaN,NaN,1947,1,1,1,0,0,pre_pandemic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1947-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.62,NaN,NaN,NaN,1947,1,2,1,0,0,pre_pandemic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1947-03-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.00,NaN,NaN,NaN,1947,1,3,1,0,0,pre_pandemic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1947-04-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.00,NaN,NaN,NaN,1947,2,4,1,0,0,pre_pandemic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1947-0